# **R/S BENCHMARK - DATASET GENERATION**

## **1. State limit function**

$$g = k(t) \cdot \frac{R}{z_1} - S \cdot z_2$$

$z_1$ and $z_2$ are normal latent multipliers (mean 1.0, sd 0.028 and 0.096); $k(t) = 1 +
(k_{final}-1)\,t/100$ is the degradation factor.

## **2. Libraries**

In [1]:
import sys
import time
from pathlib import Path

# functions.py sits one directory up
sys.path.insert(0, str(Path.cwd().parent))

import dill
import numpy as np
import pandas as pd

from functions import *
from UQpy.distributions import Normal, JointIndependent

/home/casa-wand/Documentos/2024-1_victor_hugo_renata_maria/.venv/lib/python3.11/site-packages/UQpy/__init__.py:6: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


## **3. Random variables and fixed parameters**

Design variables $R$ and $S$, plus everything the emulator needs that isn't a design variable.

In [2]:
r_mean = 5.0   # resistance mean
r_std  = 0.8   # resistance standard deviation
s_mean = 2.0   # load mean
s_std  = 0.6   # load standard deviation

n_samples            = 500      # Number of design samples
n_latent_samples     = 2500     # Number of latent samples per design sample. Also the filename prefix
n_samples_validation = 250      # Number of validation samples, redrawn at every time step
n_lambdas            = 4        # Number of λs (λ1, λ2, λ3, λ4)
k_factor_final       = 0.3      # Degradation factor at t = 100. Use 1.0 for no time effect
z1_std               = 0.028    # Standard deviation of the resistance latent multiplier
z2_std               = 0.096    # Standard deviation of the load latent multiplier

times = np.linspace(0, 150, 10, endpoint=True)  # Time points for the degradation factor
times

array([  0.        ,  16.66666667,  33.33333333,  50.        ,
        66.66666667,  83.33333333, 100.        , 116.66666667,
       133.33333333, 150.        ])

## **4. Design samples**

In [3]:
r_dist = Normal(loc=r_mean, scale=r_std)
s_dist = Normal(loc=s_mean, scale=s_std)
joint  = JointIndependent(marginals=[r_dist, s_dist])

x_pce_rvs = joint.rvs(n_samples)
x_val     = joint.rvs(n_samples_validation)

print("Samples generated successfully!")
print(f"   Number of design samples: {n_samples}")
print(f"   Number of latent samples per design sample: {n_latent_samples}")
print(f"   Total simulations per time step: {(n_samples + n_samples_validation) * n_latent_samples}")

Samples generated successfully!
   Number of design samples: 500
   Number of latent samples per design sample: 2500
   Total simulations per time step: 1875000


In [4]:
x_pce_rvs

array([[ 6.20306392,  1.622876  ],
       [ 4.67108614,  1.93905963],
       [ 5.75918879,  1.50025769],
       [ 4.90375282,  1.78476138],
       [ 3.13277343,  2.04026651],
       [ 4.49393327,  2.22536844],
       [ 4.05136404,  1.80580682],
       [ 4.93016343,  2.56318111],
       [ 4.4154427 ,  2.43758852],
       [ 4.73543469,  0.41157906],
       [ 5.43277617,  1.66831884],
       [ 4.74867256,  2.44716415],
       [ 5.64306954,  1.88196414],
       [ 4.30902743,  2.22139906],
       [ 3.46288699,  0.91238548],
       [ 5.18353406,  1.50085711],
       [ 6.10978308,  1.80251992],
       [ 5.47577846,  1.41887679],
       [ 6.47548158,  1.58656402],
       [ 5.22750575,  2.52502774],
       [ 4.41627296,  2.40997545],
       [ 5.71488108,  0.88804227],
       [ 5.64008229,  0.85866718],
       [ 5.59669241,  1.52984089],
       [ 4.94357504,  2.07521695],
       [ 4.78221944,  2.41308796],
       [ 4.77166785,  2.11872325],
       [ 4.74799046,  2.74062447],
       [ 4.67250238,

## **5. Generate the dataset at each time step**

Steps:

- $g$ evaluation;
- GLD fit; and
- saving `dataset_full`/`dataset_unique` for both splits.

In [5]:
print("="*60)
print("GENERATING THE BENCHMARK DATASET")
print("="*60)

generation_results = []
for t in times:
    result = generate_dataset_at_time_benchmark(
                                                   x_train=x_pce_rvs,
                                                   x_val=x_val,
                                                   time_step=t,
                                                   n_latent_samples=n_latent_samples,
                                                   k_factor_final=k_factor_final,
                                                   z1_std=z1_std,
                                                   z2_std=z2_std,
                                                   output_dir='.',
                                               )
    generation_results.append(result)

GENERATING THE BENCHMARK DATASET

----------------------------------------
GENERATING DATASET FOR TIME STEP: 0.0 years
----------------------------------------
  train: 500 design points, 2.47 s total
  val: 250 design points, 1.25 s total

----------------------------------------
GENERATING DATASET FOR TIME STEP: 16.666666666666668 years
----------------------------------------
  train: 500 design points, 2.45 s total
  val: 250 design points, 1.26 s total

----------------------------------------
GENERATING DATASET FOR TIME STEP: 33.333333333333336 years
----------------------------------------
  train: 500 design points, 2.56 s total
  val: 250 design points, 1.21 s total

----------------------------------------
GENERATING DATASET FOR TIME STEP: 50.0 years
----------------------------------------
  train: 500 design points, 2.46 s total
  val: 250 design points, 1.19 s total

----------------------------------------
GENERATING DATASET FOR TIME STEP: 66.66666666666667 years
--------

In [6]:
with open('2500_dataset_full_train_0.0_benchmark.pkl', 'rb') as f:
    obj = dill.load(f)
obj.head()

,r,s,z1_latent,R_effective,z2_latent,S_effective,k factor,Time (years),g,lambda 1,lambda 2,lambda 3,lambda 4,Processing time (s)
0,6.203064,1.622876,1.018524,6.090251,0.966348,1.568262,1.0,0.0,4.521989,4.584472,6.235749,0.143839,0.135863,0.007457
1,6.203064,1.622876,1.009199,6.146521,1.075439,1.745305,1.0,0.0,4.401216,4.584472,6.235749,0.143839,0.135863,0.007457
2,6.203064,1.622876,0.984255,6.302295,1.064698,1.727874,1.0,0.0,4.574421,4.584472,6.235749,0.143839,0.135863,0.007457
3,6.203064,1.622876,1.038136,5.975194,0.819673,1.330228,1.0,0.0,4.644966,4.584472,6.235749,0.143839,0.135863,0.007457
4,6.203064,1.622876,1.034313,5.997277,0.925459,1.501905,1.0,0.0,4.495372,4.584472,6.235749,0.143839,0.135863,0.007457


## **6. Timing summary**

Cost of building the dataset, per time step.

In [7]:
timing_rows = []
for result in generation_results:
    train_t = result['df_unique_train']['Processing time (s)']
    val_t   = result['df_unique_val']['Processing time (s)']
    timing_rows.append({
                           'Time (years)':   result['time_step'],
                           'n_train':        len(train_t),
                           'Train total (s)': train_t.sum(),
                           'Train mean (ms)': train_t.mean() * 1e3,
                           'n_val':          len(val_t),
                           'Val total (s)':  val_t.sum(),
                       })

emulator_timing = pd.DataFrame(timing_rows)

with open(f'{n_latent_samples}_emulator_timing_benchmark.pkl', 'wb') as f:
    dill.dump(emulator_timing, f)

train_total = emulator_timing['Train total (s)'].sum()
val_total   = emulator_timing['Val total (s)'].sum()

print(f"Train split - eg-value dataset generation time: {train_total:.1f} s")
print(f"Val split   - g-value dataset generation time: {val_total:.1f} s")
print(f"Total g-value dataset generation time (train + val): {train_total + val_total:.1f} s")
emulator_timing

Train split - eg-value dataset generation time: 24.1 s
Val split   - g-value dataset generation time: 11.8 s
Total g-value dataset generation time (train + val): 35.9 s


,Time (years),n_train,Train total (s),Train mean (ms),n_val,Val total (s)
0,0.000000,500,2.471570,4.943139,250,1.253961
1,16.666667,500,2.445753,4.891506,250,1.258739
2,33.333333,500,2.564012,5.128024,250,1.214850
3,50.000000,500,2.461701,4.923402,250,1.188311
4,66.666667,500,2.341963,4.683925,250,1.143019
5,83.333333,500,2.333950,4.667900,250,1.133430
6,100.000000,500,2.434830,4.869659,250,1.159812
7,116.666667,500,2.337434,4.674867,250,1.172869
8,133.333333,500,2.399430,4.798861,250,1.128911
9,150.000000,500,2.313289,4.626577,250,1.149276
